In [0]:
%run ../../Project_SetUp

In [0]:
%run ../common/01_common_silver_cleaner

In [0]:
%run ../common/02_common_column_mapping

In [0]:
%run ../common/03_common_pipeline_logging

In [0]:
from datetime import datetime, timezone

pipeline_start = datetime.now(timezone.utc)

source_count = 0
processed_count = 0
rejected_count = 0
duplicate_count = 0

customer_updates_mapping = {}

try:
    customerUpdates_df = spark.table("retail360.bronze.customer_updates")

    source_count = customerUpdates_df.count()

    customerUpdates_mapped_df = dynamic_column_mapping(
        customerUpdates_df,
        customer_updates_mapping
    )

    customerUpdates_silver_df = common_silver_cleaner(
        customerUpdates_mapped_df
    )

    processed_count = customerUpdates_silver_df.count()

    write_pipeline_log(
        pipeline_name="silver_customer_updates",
        source_table="retail360.bronze.customer_updates",
        target_table="retail360.silver.customer_updates",
        layer_name="SILVER",
        start_timestamp=pipeline_start,
        execution_status="SUCCESS",
        source_count=source_count,
        success_count=processed_count,
        error_message=""
    )

except Exception as e:

    write_pipeline_log(
        pipeline_name="silver_customer_updates",
        source_table="retail360.bronze.customer_updates",
        target_table="retail360.silver.customer_updates",
        layer_name="SILVER",
        start_timestamp=pipeline_start,
        execution_status="FAILED",
        source_count=source_count,
        success_count=processed_count,
        error_message=str(e)[:4000]
    )

    raise

In [0]:
%sql
SELECT
    pipeline_name,
    execution_status,
    COUNT(*) AS total_runs,
    SUM(source_count) AS total_source_records,
    SUM(success_count) AS total_success_records,
    SUM(rejected_count) AS total_rejected_records,
    SUM(duplicate_count) AS total_duplicate_records,
    SUM(error_count) AS total_errors,
    ROUND(AVG(duration_seconds), 2) AS avg_duration_seconds
FROM retail360.silver.pipeline_execution_log
GROUP BY
    pipeline_name,
    execution_status
ORDER BY pipeline_name;